## Task: Sentiment Analysis Using Lexicon-Based and Machine Learning Approaches

###  Objective

Your goal is to perform sentiment analysis on a real-world review dataset using two distinct approaches:
1. **Lexicon-based approach using TextBlob**
2. **Machine learning-based approach using Naive Bayes SVM and Randomforest**

You will compare the performance of both methods in term of accuracy, precision, recall, and f-1 score

---

### Dataset

Use the provided `All_Beauty_5.json` dataset (Amazon product reviews). Each entry contains:
- `reviewText`: the content of the review
- `overall`: the product rating (1–5 stars)







In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
import pandas as pd

# Load as JSON lines (each line is a JSON object)
df = pd.read_json("/content/drive/MyDrive/INFO 585/All_Beauty_5.json", lines=True)

# Select relevant columns
df = df[['reviewText', 'overall']].dropna()
df = df.rename(columns={'reviewText': 'text', 'overall': 'rating'})

# Convert rating to sentiment label
def rating_to_label(r):
    if r >= 4:
        return 'positive'
    elif r <= 2:
        return 'negative'
    else:
        return 'neutral'

df['label'] = df['rating'].apply(rating_to_label)
df = df[df['label'] != 'neutral']  # For binary classification

df.head()


,text,rating,label
0,As advertised. Reasonably priced,5,positive
1,Like the oder and the feel when I put it on my...,5,positive
2,I bought this to smell nice after I shave. Wh...,1,negative
3,HEY!! I am an Aqua Velva Man and absolutely lo...,5,positive
4,If you ever want to feel pampered by a shampoo...,5,positive


Lexicon Method

In [34]:
from textblob import TextBlob

def get_textblob_sentiment(text):
  polarity = TextBlob(text).sentiment.polarity
  if polarity > 0:
    return 'positive'
  elif polarity < 0:
    return 'negative'
  else:
    return 'neutral'

df['tb_sentiment'] = df['text'].apply(get_textblob_sentiment)

Evaluate

In [35]:
from sklearn.metrics import classification_report, accuracy_score

print("TextBlob Accuracy:", accuracy_score(df['label'], df['tb_sentiment']))
print(classification_report(df['label'], df['tb_sentiment']))

TextBlob Accuracy: 0.8935014548981571
              precision    recall  f1-score   support

    negative       0.27      0.30      0.28       179
     neutral       0.00      0.00      0.00         0
    positive       0.98      0.91      0.95      4976

    accuracy                           0.89      5155
   macro avg       0.42      0.41      0.41      5155
weighted avg       0.95      0.89      0.92      5155



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TextBlob works by using a predefined dictionary of words to determine whether a review is positive or negative. It performed fairly well with an accuracy of around 0.89, but it struggled when it came to detecting negative reviews. This is clear from the low precision and recall scores for the negative class. This shows that TextBlob has difficulty understanding context and certain phrases, since it relies only on predefined word lists instead of learning patterns from the data.The results also show class imbalance, since there are significantly more positive reviews than negative ones, which affects the evaluation metrics for the negative class.

Machine Learning Models

TF-IDF

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['text'])
y = df['label']

Train/Test split

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Naive Bayes

In [38]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9641125121241513
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        37
    positive       0.96      1.00      0.98       994

    accuracy                           0.96      1031
   macro avg       0.48      0.50      0.49      1031
weighted avg       0.93      0.96      0.95      1031



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


SVM

In [39]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9951503394762367
              precision    recall  f1-score   support

    negative       1.00      0.86      0.93        37
    positive       0.99      1.00      1.00       994

    accuracy                           1.00      1031
   macro avg       1.00      0.93      0.96      1031
weighted avg       1.00      1.00      0.99      1031



Random Forest

In [40]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.9951503394762367
              precision    recall  f1-score   support

    negative       1.00      0.86      0.93        37
    positive       0.99      1.00      1.00       994

    accuracy                           1.00      1031
   macro avg       1.00      0.93      0.96      1031
weighted avg       1.00      1.00      0.99      1031



The results show that the machine learning models performed better than the lexicon-based approach for sentiment classification. While TextBlob had an accuracy of about 0.89, it had trouble identifying negative reviews, which we can see from the low precision and recall for the negative class. This happens because lexicon-based methods rely on predefined word lists and do not fully understand context.

On the other hand, the machine learning models such as Naive Bayes, SVM, and Random Forest had higher accuracy scores, with SVM and Random Forest performing the best at around 0.99 accuracy. These models are able to learn patterns from the data, which makes them more effective at capturing sentiment.

However, machine learning models require labeled data and more preprocessing, while TextBlob is simpler and faster to use. Overall, machine learning provides better performance, but lexicon-based methods can still be useful for quick and simple analysis.